In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='others/baseline/HitScreen/script')
_ks_sys.path.insert(0, str(KINSPEC_ROOT / 'external/HitScreen'))
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'others/baseline/HitScreen/script')



In [1]:
# launch_all_training_sequential.py
import subprocess
import sys
from pathlib import Path
from datetime import datetime
import time

# Flush output
sys.stdout.reconfigure(line_buffering=True)
sys.stderr.reconfigure(line_buffering=True)

# Set the base path
base_path = Path(_kp('dataset/data_split_from_merged_true_fold'))
main_script = Path(_kp('external/HitScreen/main.py'))

# Define the band ranges
bands = [
    "band_0.10_0.50",
    # "band_0.20_0.60",
    # "band_0.30_0.70",
    # "band_0.40_0.80",
    # "band_0.50_0.90",
    # "band_0.60_1.00"
]

# Define train_valid directories
train_valid_folders = [f"train_valid_split/train_valid_{i}" for i in range(1, 6)]

def log_message(message):
    """Write to the log and print, flushing output immediately."""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = f"[{timestamp}] {message}"
    print(log_entry, flush=True)

def format_time(seconds):
    """Format a duration."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"{hours}h {minutes}m {secs}s"
    elif minutes > 0:
        return f"{minutes}m {secs}s"
    else:
        return f"{secs}s"

# Start processing
log_message("=" * 80)
log_message("Starting HitScreen Training Process (SEQUENTIAL MODE)")
log_message("Each task will complete before the next one starts")
log_message(f"Base path: {base_path}")
log_message(f"Main script: {main_script}")
log_message(f"Total tasks to process: {len(bands) * len(train_valid_folders)}")
log_message("=" * 80)

total_count = 0
success_count = 0
failed_count = 0
skipped_count = 0
total_start_time = time.time()

# Iterate over all band and train_valid combinations
for band in bands:
    log_message(f"\n{'='*80}")
    log_message(f"Processing band: {band}")
    log_message(f"{'='*80}")
    
    band_nums = band.replace("band_", "").replace(".", "")
    
    for train_valid in train_valid_folders:
        total_count += 1
        tv_num = train_valid.split('_')[-1]
        
        # Build paths
        data_path = base_path / band / train_valid
        result_dir = KINSPEC_ROOT / "model/HitScreen/data_split_from_merged_true_fold" / band / "train_valid_split" / train_valid / "result"
        
        # Generate the log filename
        date_str = datetime.now().strftime('%y%m%d')
        task_log_file = data_path / f"hitscreen_{band_nums}_tv{tv_num}_{date_str}.log"
        
        # Calculate elapsed time and estimate remaining time
        elapsed_total = time.time() - total_start_time
        if success_count > 0:
            avg_time = elapsed_total / success_count
            remaining_tasks = len(bands) * len(train_valid_folders) - total_count + 1
            estimated_remaining = avg_time * remaining_tasks
            log_message(f"\n[Task {total_count}/{len(bands) * len(train_valid_folders)}] Progress: {total_count-1}/{len(bands) * len(train_valid_folders)} completed")
            log_message(f"  Elapsed: {format_time(elapsed_total)} | Estimated remaining: {format_time(estimated_remaining)}")
        else:
            log_message(f"\n[Task {total_count}/{len(bands) * len(train_valid_folders)}]")
        
        log_message(f"  Band: {band} | Train-Valid: {train_valid}")
        log_message(f"  Data path: {data_path}")
        log_message(f"  Task log:  {task_log_file}")
        
        # Check whether data paths exist
        if not data_path.exists():
            log_message(f"  ✗ SKIPPED: Data path does not exist")
            skipped_count += 1
            continue
        
        # Create the results directory
        try:
            result_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            log_message(f"  ✗ FAILED to create result directory: {e}")
            failed_count += 1
            continue
        
        # Check whether training files exist
        train_file = data_path / "train_augment.csv"
        if not train_file.exists():
            train_file = data_path / "train.csv"
            if not train_file.exists():
                log_message(f"  ✗ SKIPPED: No train.csv or train_augment.csv found")
                skipped_count += 1
                continue
        
        log_message(f"  ✓ Training file: {train_file.name}")
        
        # Build the command
        cmd = [
            _ks_sys.executable, str(main_script),
            "--data", str(data_path),
            "--result_output_dir", str(result_dir) + "/",
            "--result_save_model", str(result_dir) + "/"
        ]
        
        try:
            log_message(f"  ▶ Starting training...")
            log_message(f"  Command: {' '.join(cmd)}")
            log_message(f"  Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            
            task_start_time = time.time()
            
            # Run training and redirect output to the log
            with open(task_log_file, 'w') as log_f:
                log_f.write(f"Training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                log_f.write(f"Command: {' '.join(cmd)}\n")
                log_f.write("=" * 80 + "\n\n")
                log_f.flush()
                
                process = subprocess.Popen(
                    cmd,
                    stdout=log_f,
                    stderr=subprocess.STDOUT,
                    bufsize=1,
                    universal_newlines=True
                )
                
                # Wait for the process to complete
                return_code = process.wait()
            
            task_elapsed_time = time.time() - task_start_time
            
            if return_code == 0:
                log_message(f"  ✓ COMPLETED successfully")
                log_message(f"  Duration: {format_time(task_elapsed_time)}")
                log_message(f"  Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                success_count += 1
            else:
                log_message(f"  ✗ FAILED with return code {return_code}")
                log_message(f"  Duration: {format_time(task_elapsed_time)}")
                log_message(f"  Check log: {task_log_file}")
                failed_count += 1
                
        except KeyboardInterrupt:
            log_message(f"\n{'='*80}")
            log_message(f"⚠ INTERRUPTED by user (Ctrl+C)")
            log_message(f"{'='*80}")
            log_message(f"Completed: {success_count}/{total_count} tasks")
            log_message(f"Failed: {failed_count}")
            log_message(f"Skipped: {skipped_count}")
            log_message(f"Interrupted at task: {band}/{train_valid}")
            sys.exit(1)
            
        except Exception as e:
            log_message(f"  ✗ FAILED with exception: {str(e)}")
            failed_count += 1

# Print final aggregate results
total_elapsed = time.time() - total_start_time

log_message("\n" + "=" * 80)
log_message("FINAL TRAINING SUMMARY")
log_message("=" * 80)
log_message(f"Total tasks:       {total_count}")
log_message(f"Successful:        {success_count}")
log_message(f"Failed:            {failed_count}")
log_message(f"Skipped:           {skipped_count}")
log_message(f"Success rate:      {success_count/total_count*100:.2f}%" if total_count > 0 else "N/A")
log_message(f"Total time:        {format_time(total_elapsed)}")
if success_count > 0:
    log_message(f"Average per task:  {format_time(total_elapsed/success_count)}")
log_message("=" * 80)
log_message("All training jobs completed!")
log_message(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_message("=" * 80)

AttributeError: 'OutStream' object has no attribute 'reconfigure'